# R2 pipeline — Phase-I launch (thin driver)

Spec §4.8: notebooks are thin drivers, not where logic lives. This one (1) resolves paths
and provenance, (2) launches the R2 Phase-I ignition run under the memory guard
(`CLAUDE.md` §7a — never a bare trainer call), and (3) summarises the run's own
`results/*.json` once it has finished. It contains no analysis and defines no helper
functions; every real computation lives in `src/rngrn/` or `scripts/r2_ignition_run.py`.

**Not executed as committed.** `scripts/r2_ignition_run.py` is Task 16's deliverable and
does not exist on this branch yet, so the launch cell in this notebook has never been run —
running it before that script lands fails loud with a missing-script error, which is the
correct behaviour, not a bug to work around here.

In [ ]:
import glob
import json
import os
import subprocess
import sys

REPO = (os.path.abspath(os.path.join(os.getcwd(), ".."))
        if os.path.basename(os.getcwd()) == "notebooks"
        else os.path.abspath(os.getcwd()))
sys.path.insert(0, os.path.join(REPO, "src"))

PHASE1_OUT = os.path.join(REPO, "experiments", "redesign_r2", "phase1")

print("repo:      ", REPO)
print("phase1 out:", PHASE1_OUT)

## Provenance

CLAUDE.md §2: the venv actually importing `rngrn` from *this* worktree, not a sibling one, is what makes every other cell's result trustworthy.

In [ ]:
import rngrn
from rngrn.utils import provenance

prov = provenance()
rngrn_path = os.path.dirname(rngrn.__file__)

print("git SHA:    ", prov["git_revision"])
print("python:     ", prov["python"])
print("torch:      ", prov.get("torch"))
print("device:     ", prov.get("device"))
print("rngrn path: ", rngrn_path)
print("repo root:  ", REPO)

assert rngrn_path.startswith(REPO), (
    "rngrn imports from outside this worktree (CLAUDE.md §2 editable-install gotcha) — "
    f"got {rngrn_path!r}, expected it under {REPO!r}")

## Phase-I launch (guarded)

`bash scripts/guarded_run.sh` (`CLAUDE.md` §7a) is the only way a trainer runs in this repo — it serialises against every other worktree's trainers under one `flock` and raises its own OOM-kill priority so a memory spike takes down the trainer, not the session.

In [ ]:
cmd = ["bash", "scripts/guarded_run.sh", ".venv/bin/python",
       "scripts/r2_ignition_run.py", "--out", "experiments/redesign_r2/phase1"]
print("launching:", " ".join(cmd))
subprocess.run(cmd, cwd=REPO, check=True)

## Results summary

Reads whatever `results/*.json` the run wrote — the notebook does not assume a fixed schema, since the exact result file(s) are `scripts/r2_ignition_run.py`'s (Task 16) to define.

In [ ]:
results_dir = os.path.join(PHASE1_OUT, "results")
result_files = sorted(glob.glob(os.path.join(results_dir, "*.json")))
print(f"{len(result_files)} result file(s) under {results_dir}")

for path in result_files:
    with open(path) as f:
        summary = json.load(f)
    print("---", os.path.basename(path), "---")
    print(json.dumps(summary, indent=2, default=str))